# Rally E4B RP Text Export

Exports the gate-validated A100/B75 RP text browser package from `rally-e4b-sft-jun14v10` using the optimized Gemma4 E4B q4f16 template.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token:
    os.environ.setdefault('HF_TOKEN', secret_token)
    os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', secret_token)
packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0', 'huggingface_hub[cli]>=1.5.0', 'hf_transfer>=0.1.9',
    'safetensors>=0.7.0', 'onnx>=1.19.0', 'onnxruntime>=1.23.0', 'onnxscript>=0.5.0',
    'sentencepiece>=0.2.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)

prep_candidates = [
    Path('/kaggle/input/rally-e4b-export-prep/rally-e4b-export-prep'),
    Path('/kaggle/input/rally-e4b-export-prep'),
    Path('/kaggle/input/notebooks/thomasjvu/rally-e4b-export-prep/rally-e4b-export-prep'),
    Path('/kaggle/input/notebooks/thomasjvu/rally-e4b-export-prep'),
]
prep_root = first_existing(prep_candidates)
if prep_root is None:
    raise FileNotFoundError('Could not locate E4B export prep output')

staged_repo = prep_root / 'heretic-to-onnx'
staged_template = prep_root / 'optimized-template'
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(staged_repo, REPO_DIR)

WORK_DIR = Path('/kaggle/working/rally-e4b-rp-text-export')
REPORT_PATH = WORK_DIR / 'rally-e4b-browser-export-report.json'
cmd = [
    sys.executable, str(REPO_DIR / 'scripts/kaggle_rally_e2b_two_stage_export.py'),
    '--work-dir', str(WORK_DIR),
    '--report-path', str(REPORT_PATH),
    '--scratch-dir', '/kaggle/temp/rally-e4b-rp-text-export',
    '--artifact-name', os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e4b-two-stage-sft'),
    '--direct-source-model-id', os.environ.get('RALLY_HERETIC_MODEL_ID', 'coder3101/gemma-4-E4B-it-heretic'),
    '--base-model-id', os.environ.get('RALLY_BASE_MODEL_ID', 'google/gemma-4-E4B-it'),
    '--direct-full-repo', os.environ.get('RALLY_DIRECT_FULL_REPO', 'thomasjvu/rally-4b'),
    '--direct-text-repo', os.environ.get('RALLY_DIRECT_TEXT_REPO', 'thomasjvu/rally-4b-text'),
    '--rp-merged-repo', os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-4b-rp-source-merged'),
    '--rp-full-repo', os.environ.get('RALLY_RP_FULL_REPO', 'thomasjvu/rally-4b-rp'),
    '--rp-text-repo', os.environ.get('RALLY_RP_TEXT_REPO', 'thomasjvu/rally-4b-rp-text'),
    '--stage-b-scale', os.environ.get('RALLY_STAGE_B_SCALE', '0.75'),
    '--export-device', os.environ.get('RALLY_EXPORT_DEVICE', 'cpu'),
    '--optimized-template-dir', str(staged_template),
    '--optimized-template-model-id', os.environ.get('RALLY_OPTIMIZED_TEMPLATE_MODEL_ID', 'onnx-community/gemma-4-E4B-it-ONNX'),
    '--direct-full-template', 'configs/heretic-to-onnx.gemma4-e4b-heretic.yaml',
    '--direct-text-template', 'configs/heretic-to-onnx.gemma4-e4b-heretic-text.yaml',
    '--rp-full-template', 'configs/heretic-to-onnx.gemma4-e4b-heretic.yaml',
    '--rp-text-template', 'configs/heretic-to-onnx.gemma4-e4b-rp-text.yaml',
    '--skip-direct', '--skip-full-packages', '--no-score',
]
if os.environ.get('RALLY_UPLOAD', '0') != '1':
    cmd.append('--no-upload')
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json

report_path = Path('/kaggle/working/rally-e4b-rp-text-export/rally-e4b-browser-export-report.json')
report = json.loads(report_path.read_text())
print(json.dumps({
    'ok': report.get('ok'),
    'exports': [item.get('target', {}).get('repo_id') for item in report.get('exports', [])],
    'warnings': report.get('warnings', []),
}, indent=2))
